# Práctica: Fundamentos de LLMs y Configuración de la API

¡Bienvenidos a la parte práctica! En esta sesión dejaremos atrás los modelos tradicionales y experimentaremos con los Large Language Models (LLMs) directamente a través del código. Utilizaremos el modelo **Gemini de Google**, ya que su nivel gratuito ofrece una amplia capacidad.

## 1. Obtener la API Key de Gemini

Para que nuestro código en Python pueda comunicarse con el modelo de Gemini que está alojado en los servidores de Google, necesitamos unas "credenciales" únicas: la **API Key**.

### Pasos a seguir:
1. Ve a la web de **Google AI Studio**: [https://aistudio.google.com/](https://aistudio.google.com/) e inicia sesión con una cuenta de Google.
2. En el menú de la izquierda, haz clic en el botón **"Get API key"**.
3. Haz clic en **"Create API key"** y selecciona "Create API key in new project".
4. Se generará una cadena de texto larga (algo parecido a `AIzaSyB...`). **Cópiala y guárdala**. ¡No la compartas públicamente!

---

## 2. Instalación de dependencias
Vamos a instalar el SDK oficial de Google para Python, que nos facilitará enormemente interactuar con su API.

In [ ]:
!pip install -q -U google-generativeai

---
## 3. Configuración Inicial
Importamos la librería y le pasamos nuestra API KEY (Copia la que obtuviste en el paso anterior y ponla en la variable).

In [ ]:
import google.generativeai as genai

# SUSTITUYE AQUÍ TU API KEY
GOOGLE_API_KEY = "TU_API_KEY_AQUI_ENTRE_COMILLAS"

genai.configure(api_key=GOOGLE_API_KEY)

# Instanciamos el modelo. Usaremos gemini-1.5-flash ya que es la versión gratuita por defecto
model = genai.GenerativeModel('gemini-1.5-flash')

---
## 4. Peticiones Simples
Hacer una petición a la API es tan sencillo como llamar a un método. Vamos a empezar probando a pedirle algo simple.

In [ ]:
prompt_ejemplo = "¿En qué año se lanzó el primer iPhone de Apple?"
response = model.generate_content(prompt_ejemplo)

print(f"Pregunta: {prompt_ejemplo}")
print("-" * 30)
print(response.text)

### Ejercicio 1: Prompting Estructurado
El verdadero poder está en cómo le pides las cosas. 
**Tu turno:** Escribe un *prompt* que contenga varios elementos (como vimos en teoría: Rol, Tarea, Contexto, Formato). Pide, por ejemplo, que actúe como un pirata informático, que te cuente en exactamente 3 frases cuál es su lenguaje de programación favorito, y que te devuelva el resultado envuelto en etiquetas HTML `<strong>`.

In [ ]:
# EJERCICIO 1: Completa el string con tu prompt y lánzaselo al modelo. 
mi_primer_prompt = """
(Escribe aquí tu prompt estructurado...)
"""

# Tu código aquí para lanzar la consulta
# response_ej1 = ...
# print(response_ej1.text)

---
## 5. Parámetros del API: Temperature
Como comentamos en clase, el modelo no es 100% determinista. Podemos decirle mediante parámetros cómo queremos que sea de predecible o "creativo" usando el valor `temperature` (va desde 0.0 hasta 1.0+).  
Veamos cómo afecta a un mismo *prompt* lanzar la misma consulta dos veces con diferentes valores.

In [ ]:
# Vamos a pedirle un nombre creativo:
prompt_nombres = "Escribe 3 posibles nombres muy inventados y fantasiosos para un gato mágico."

for t in [0.0, 1.0]:
    config = genai.types.GenerationConfig(temperature=t)
    # Hacemos la llamada pasando la configuración
    response = model.generate_content(prompt_nombres, generation_config=config)
    print(f"\n[ TEMPERATURE: {t} ] {'-'*30}")
    print(response.text)

# Nota: ¡Ejecuta esta celda varias veces! Observa cómo los de temp 0.0 casi siempre 
#     son los mismos o muy parecidos, y los de temp 1.0 varían bastante más cada vez que la ejecutas.

### Ejercicio 2: Experimentando con Temperature
Te toca. Crea un nuevo bloque de código donde el *prompt* sea la trama inicial o comienzo de un cuento de ciencia ficción corto. Ejecuta un bucle parecido al de arriba con valores `[0.0, 0.5, 1.0, 1.5, 2.0]`.  
Verás qué pasa cuando le damos al modelo *"demasiada creatividad"* y llega a romper la coherencia del texto con temperaturas extremas.

In [ ]:
# EJERCICIO 2: Temperaturas extremas en la creatividad

# tu código aquí y tu prompt...

---
## 6. Alucinaciones (mintiendo con muchísima confianza)
Si le pedimos algo para lo que no tiene información porque es un dato local tuyo, o es muy moderno y no está en sus datos de entrenamiento, se lo inventará (para completar probabilísticamente el texto).  
Esto es uno de los problemas fundamentales, y debes comprobarlo.

In [ ]:
# Preguntemos por una persona y un evento inventados:
prompt_alucinacion = "¿Cuáles fueron las principales políticas del famoso Rey de España 'Alfonso XIV el Mago' durante su reinado en el año 2045?"
response_alucinacion = model.generate_content(prompt_alucinacion)

print(response_alucinacion.text)

### Ejercicio 3: Mitigando la Alucinación
En la teoría vimos cómo **dar contexto** y **poner restricciones estrictas** puede evitar esto.
**Tu turno:** Modifica el prompt de arriba (o crea uno nuevo) sobre el mismo tema ("El rey Alfonso XIV en 2045") pero aplicando técnicas de mitigación para que el modelo te responda que no lo sabe o que esa información no existe, en lugar de inventársela.

In [ ]:
# EJERCICIO 3: Crea un prompt que mitigue la alucinación

prompt_mitigado = """
(Ej: Eres un historiador riguroso. Vas a responder a mis preguntas basándote sólo en...)
"""

# Tu código para llamar al modelo con este nuevo prompt
# print(resultado...)

---
## 7. El LLM como herramienta "Todo en Uno"

En la práctica anterior con Hugging Face, necesitabas cargar **un pipeline y un modelo distinto** para cada tarea (analizar sentimiento, traducir, resumir, extraer entidades...). 

Con un LLM puedes hacer *todo* eso a la vez con un solo prompt. ¡Vamos a estresarlo!

### Ejercicio 4: Traducción y Resumen simultáneo
En un solo prompt, pásale el texto en inglés de abajo, pídele que extraiga sus **3 puntos clave**, y que te los devuelva directamente **traducidos al español**.

In [ ]:
# EJERCICIO 4: Traducción y Resumen en un solo paso

texto_ingles = """
The James Webb Space Telescope (JWST) is an infrared space observatory that launched on Dec 25, 2021. 
It is the largest and most powerful space telescope ever built. 
Its main goals are to study the first stars and galaxies that formed after the Big Bang, 
to observe planetary systems around other stars, and to explore the potential for life on exoplanets. 
Unlike the Hubble telescope, which orbits Earth, JWST orbits the Sun at the second Lagrange point (L2), 
keeping it permanently shaded from the Sun to maintain its ultra-cold operating temperature.
"""

prompt_todo_en_uno = f"""
(Escribe aquí tu prompt pidiendo: Extrae 3 puntos clave en español del texto provisto)

Texto:
{texto_ingles}
"""

# Tu código para llamar al modelo e imprimir el resultado:
# response_ej4 = ...
# print(response_ej4.text)

---
### Ejercicio 5: Explicación y Generación de Código
Los modelos de lenguaje son excepcionales en tareas de programación. Una de sus aplicaciones estrella no es solo crear código de cero, sino **explicar, refactorizar y encontrar errores lógicos** o de seguridad (bugs) en código ya existente.

**Tu turno:** El siguiente fragmento de código Python está mal escrito, es confuso y contiene errores lógicos importantes (por ejemplo, permitir dividir por cero, falta manejo de tipos y convenciones). Diseña un prompt que evalúe y limpie esta función: pide a Gemini que te **explique brevemente qué intenta hacer** y después te devuelva el **código refactorizado en Python limpio**.

In [ ]:
# EJERCICIO 5: Refactorizar y explicar código sucio

codigo_sucio = """
def clc(x,y, o):
  if o=='A': return x+y
  if o=='B': 
    print(x-y)
    return x-y
  elif o=='C': return x*y
  elif o=='D': return x/y # podria haber error si y es 0 pero bueno no pasa na
  else:
        return 'no me se esto'
"""

prompt_codigo = f"""
(Escribe aquí tu prompt pidiendo: explicar qué hace y presentar la versión en código limpio)

Código:
{codigo_sucio}
"""

# Tu código para llamar al modelo e imprimir el resultado:
# response_ej5 = ...
# print(response_ej5.text)

---
### Ejercicio 6: Análisis de Sentimiento y Clasificación Compleja (Combo)

Anteriormente hicimos análisis de sentimiento en 1 pipeline y Extracción de Entidades Nombradas (NER) en otro diferente. Con un LLM puedes decirle en el mismo prompt que clasifique el sentimiento genérico y además extraiga datos precisos bajo el formato que tú elijas.

**Tu turno:** Diseña un prompt para que el modelo lea una review de un comercio online, clasifique el sentimiento general (Positivo, Neutro o Negativo), e indique de forma separada el motivo de la queja / alabanza principal.

In [ ]:
# EJERCICIO 6: Clasificación y Extracción mixta

review_cliente = "Pagué por envío urgente 24h para las zapatillas, y no me llegaron hasta 5 días después perdiendo el evento. Es cierto que se ven muy bien y la talla me cuadra, pero el servicio postventa ni siquiera contesta para compensarme."

prompt_review = f"""
(Tu instrucción pidiendo: Sentimiento = NEGATIVO/POSITIVO/NEUTRO, Queja principal = 'Motivo...')

Review:
{review_cliente}
"""

# Tu código aquí
# response_ej6 = ...
# print(response_ej6.text)

---
## 8. El modo Chat (Memoria en la Conversación)

Hasta ahora hemos usado peticiones *aisladas*: llamamos a `generate_content()`, la API recibe nuestra pregunta, emite una respuesta, y **todo se olvida**. No hay ninguna "memoria".

¿Y si queremos poder hacerle repreguntas, tal y cómo hacemos al chatear con ChatGPT, donde se acuerda de lo que comentamos arriba? 
Para eso, emplearemos la función nativa de iniciar sesión de **Chat** incluida en el SDK de Google. Ella sola guarda el historial y se lo reenvía escondido al modelo.

### Ejercicio 7: Chateando con el modelo
Acá simularemos enviar un primer mensaje de presentación en el que damos dos puntos de información. En la segunda consulta (en la que NO volvemos a repetir esa información), le pediremos al modelo que la recuerde. Para que este ejercicio funcione, tienes que enviar el mensaje usando un objeto especial `.send_message()` en vez de la función general del modelo base.

In [ ]:
# EJERCICIO 7: Iniciar una sesión mantenida de Chat (con Historial)

# 1. Instanciamos una sesión nueva de chat, que arranca siempre con la memoria vacía (=historial vacío)
chat = model.start_chat()

# 2. Enviamos el primer mensaje usando el método especial '.send_message' envés de .generate_content() del modelo base.
primer_mensaje = "Hola, me llamo Juanito y mi plato preferido de comer son las croquetas."

print(f"Yo: {primer_mensaje}")
respuesta_1 = chat.send_message(primer_mensaje)
print(f"Gemini: {respuesta_1.text}\n")


# 3. Mandamos un TERCER mensaje que requiera recordar el primer mensaje arriba que le mandamos para contestar esto.
segundo_mensaje = "¡Qué tal! Escucha una cosilla, recuérdame ¿cómo me llamo yo? ¿y qué es lo que quiero para cenar hoy?"

print(f"Yo: {segundo_mensaje}")

# Tu turno: manda el segundo mensaje usando `chat.send_message()` usando la variable segundo_mensaje
# y verifica que SÍ SABE cómo te llamas y qué comer.
# respuesta_2 = ...
# print(f"Gemini: {...}")